# H2 分子势能面计算

本 notebook 计算 H2 分子在不同键长下的 FCI 和 HF 能量，绘制势能面曲线。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf, fci
from tqdm import tqdm

bond_distance = np.round(np.linspace(0.4, 4.0, 37), 2)

## 计算 FCI 和 HF 能量

In [ ]:
H2_E_FCI = []
H2_E_HF = []
for bond_length in tqdm(bond_distance):
    geometry = [
        ('H', (0., 0., 0.)),
        ('H', (bond_length, 0., 0.)),
    ]

    mol = gto.M(atom=geometry, basis='STO-3G')

    mf = scf.RHF(mol).run(verbose=0)
    E_hf = mf.e_tot

    cisolver = fci.FCI(mf)
    E_fci, fcivec = cisolver.kernel()
    H2_E_FCI.append(E_fci)
    H2_E_HF.append(E_hf)
    
with open('H2_E_FCI_HF.npz', 'wb') as f:
    np.savez(f, E_FCI=H2_E_FCI, E_HF=H2_E_HF, bond_length=bond_distance)

## 绘制势能面

In [ ]:
with open('H2_E_FCI_HF.npz', 'rb') as f:
    data = np.load(f)
    E_FCI = data['E_FCI']
    E_HF = data['E_HF']
    bond_length = data['bond_length']

plt.figure(figsize=(10, 6))
plt.plot(bond_length, E_FCI, 'b-', label='FCI energy', linewidth=2)
plt.plot(bond_length, E_HF, 'r--', label='HF energy', linewidth=2)
plt.xlabel('Bond Length (Å)', fontsize=12)
plt.ylabel('Energy (Hartree)', fontsize=12)
plt.title('H2 Molecule Potential Energy Surface', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"FCI 最小能量: {np.min(E_FCI):.6f} Hartree")
print(f"HF 最小能量: {np.min(E_HF):.6f} Hartree")
print(f"FCI 平衡键长: {bond_length[np.argmin(E_FCI)]:.2f} Å")
print(f"HF 平衡键长: {bond_length[np.argmin(E_HF)]:.2f} Å")

## 绘制相对于最小能量的势能面

In [ ]:
E_FCI_relative = E_FCI - np.min(E_FCI)
E_HF_relative = E_HF - np.min(E_HF)

plt.figure(figsize=(10, 6))
plt.plot(bond_length, E_FCI_relative, 'b-', label='FCI energy', linewidth=2)
plt.plot(bond_length, E_HF_relative, 'r--', label='HF energy', linewidth=2)
plt.xlabel('Bond Length (Å)', fontsize=12)
plt.ylabel('Relative Energy (Hartree)', fontsize=12)
plt.title('H2 Molecule Potential Energy Surface (Relative to Minimum)', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()